# Sentiment Pipeline Comparison

This notebook runs the same weekly walk-forward comparison used by the CLI experiment for:

- `finbert_sentiment`
- `lm_sentiment`
- `rule_based_directional`
- optional `combined_all`

It reuses the shared helper module in `src/features/sentiment_pipeline_comparison.py`, so the notebook and script stay aligned.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.features.sentiment_pipeline_comparison import (
    DISPLAY_COLUMNS,
    run_sentiment_pipeline_comparison,
)

ROOT

In [ ]:
FEATURE_DIR = ROOT / "data" / "features"
FINBERT_PATH = FEATURE_DIR / "finbert_feature_matrix.parquet"
LM_PATH = FEATURE_DIR / "lm_feature_matrix.parquet"
RULE_BASED_PATH = FEATURE_DIR / "rule_based_topic_feature_matrix.parquet"

MIN_TRAIN_SIZE = 156
STEP = 1
INCLUDE_COMBINED = True

pd.DataFrame(
    {
        "artifact": ["FinBERT", "LM-S", "Rule-based"],
        "path": [FINBERT_PATH, LM_PATH, RULE_BASED_PATH],
        "exists": [FINBERT_PATH.exists(), LM_PATH.exists(), RULE_BASED_PATH.exists()],
    }
)

In [ ]:
comparison, backtests, configs = run_sentiment_pipeline_comparison(
    finbert_path=FINBERT_PATH,
    lm_path=LM_PATH,
    rule_based_path=RULE_BASED_PATH,
    output_dir=FEATURE_DIR,
    min_train_size=MIN_TRAIN_SIZE,
    step=STEP,
    include_combined=INCLUDE_COMBINED,
)

comparison[DISPLAY_COLUMNS]

In [ ]:
summary = []
for name, bt in backtests.items():
    summary.append(
        {
            "pipeline": name,
            "oos_rows": len(bt),
            "oos_start": pd.to_datetime(bt["date"]).min().date().isoformat(),
            "oos_end": pd.to_datetime(bt["date"]).max().date().isoformat(),
        }
    )

pd.DataFrame(summary).sort_values("pipeline")

## Outputs

Running the notebook saves the same artifacts as the CLI script into `data/features/`:

- `sentiment_pipeline_comparison.csv`
- `finbert_sentiment_comparison_backtest.parquet`
- `lm_sentiment_comparison_backtest.parquet`
- `rule_based_directional_comparison_backtest.parquet`
- `combined_all_comparison_backtest.parquet` when `INCLUDE_COMBINED = True`
